# Run the Forge on a free GPU

Works on Kaggle and Colab. Pick a project in the second cell, then run everything.
On Kaggle, switch on the GPU and internet in the session options.

For `asr`, add your Mozilla Data Collective key as a secret named `MDC_API_KEY`
(Kaggle: Add-ons → Secrets; Colab: the key icon). Don't paste keys into cells: notebooks get shared.


In [ ]:
!git clone --depth 1 https://github.com/BarisAtik/argus-panoptes.git
%cd argus-panoptes/models
!pip install -q -e ".[dev,asr,text,vision]"
!python -m pytest -q


In [ ]:
PROJECT = "text"  # "text", "asr" or "clip"


In [ ]:
import os


def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except ImportError:
        from google.colab import userdata
        return userdata.get(name)


if PROJECT == "text":
    !python -m text_nl.cli --out runs/text
elif PROJECT == "asr":
    os.environ["MDC_API_KEY"] = secret("MDC_API_KEY")
    !python -m asr_nl.fetch_common_voice --n 2000 --out runs/cv_nl
    !python -m asr_nl.cli audit --manifest runs/cv_nl/manifest_test_2000.jsonl
    !python -m asr_nl.cli run --config asr_nl/configs/full_gpu.yaml
elif PROJECT == "clip":
    !python -m clip_search.coco --out runs/coco
    !python -m clip_search.cli --coco runs/coco --out runs/clip --model hf:openai/clip-vit-base-patch32 --model openclip:xlm-roberta-base-ViT-B-32:laion5b_s13b_b90k


Download the report before the session ends (the checkpoint is large, the report isn't):


In [ ]:
import glob
from pathlib import Path

for path in glob.glob("runs/**/report.md", recursive=True):
    print(path)
    print(Path(path).read_text()[:4000])
